# OLAP · 05 · BQ5 · Do lower-income states have lower insured loss per person?
**Owner: M** · Reviewer: R · Decision supported: **product and affordability strategy** (catastrophe pricing analyst)

If lower-income states showed much lower insured loss per resident for similar hazard exposure, that would hint at **underinsurance**: the damage happens, but less of it is insured.

| Query | Question | OLAP operation |
|---|---|---|
| 5.1 | Loss across every combination of state and hazard family | **Cube** (`GROUP BY CUBE`): all subtotals in both directions |
| 5.2 | Income, loss per resident and claims per 1,000 residents by state | **Drill-across** both facts |
| 5.3 | Correlation, with and without the most extreme state | Statistics on the drill-across result |

**Caution:** there are only eight states and territories. A correlation from eight points is a signal to investigate, not proof, and one state can flip it. Query 5.3 tests exactly that.

In [ ]:
%run ./00_olap_setup

### 5.1 · Cube: state × hazard family
`CUBE (a, b)` returns (a, b), (a), (b) and the grand total, so the table can be read by row (each state's hazard mix) or by column (each hazard's state mix).

In [ ]:
q51 = olap("bq5_cube_state_family", """
SELECT CASE grouping_id(state_code, hazard_family) WHEN 0 THEN 'state × family' WHEN 1 THEN 'state total'
                                                   WHEN 2 THEN 'family total' ELSE 'grand total' END AS level,
       coalesce(state_code, 'All states') AS state_code, coalesce(hazard_family, 'All families') AS hazard_family,
       count(*) AS events, round(sum(loss_m), 1) AS loss_m,
       grouping_id(state_code, hazard_family) AS gid
FROM v_event_loss
GROUP BY CUBE (state_code, hazard_family)
ORDER BY gid, state_code, loss_m DESC NULLS LAST
""")

### 5.2 · Drill-across: income vs insured loss and claims per resident
Claims per 1,000 residents is a second view of insurance take-up: it counts policies that claimed, not dollars.

In [ ]:
q52 = olap("bq5_income_vs_loss", """
WITH sy AS (SELECT state_code, avg(median_income_aud) AS median_income, avg(population) AS population,
                   bool_or(income_imputed) AS income_partly_imputed
            FROM v_state_year GROUP BY state_code),
ev AS (SELECT state_code, count(*) AS events, coalesce(sum(loss_m), 0) AS loss_m, coalesce(sum(total_claims), 0) AS claims
       FROM v_event_loss GROUP BY state_code)
SELECT sy.state_code, round(sy.median_income, 0) AS avg_median_income_aud,
       coalesce(ev.events, 0) AS events,
       round(1e6 * coalesce(ev.loss_m, 0) / sy.population, 0) AS loss_per_resident_aud,
       round(1000 * coalesce(ev.claims, 0) / sy.population, 1) AS claims_per_1000_residents,
       rank() OVER (ORDER BY sy.median_income) AS income_rank,
       sy.income_partly_imputed
FROM sy LEFT JOIN ev ON ev.state_code = sy.state_code
ORDER BY avg_median_income_aud
""")

### 5.3 · Correlation and a sensitivity test
`corr()` is Pearson's r. The second row drops the state with the highest loss per resident to see whether one state drives the result.

In [ ]:
q53 = olap("bq5_correlation", """
WITH s AS (SELECT * FROM {t}),
ext AS (SELECT max_by(state_code, loss_per_resident_aud) AS top_loss, max_by(state_code, avg_median_income_aud) AS top_income FROM s)
SELECT 1 AS ord, 'all states' AS sample, count(*) AS states,
       round(corr(avg_median_income_aud, loss_per_resident_aud), 2) AS r_income_vs_loss_per_resident,
       round(corr(avg_median_income_aud, claims_per_1000_residents), 2) AS r_income_vs_claims_per_1000
FROM s
UNION ALL
SELECT 2, concat('without ', first(top_loss), ' (highest loss per resident)'), count(*),
       round(corr(avg_median_income_aud, loss_per_resident_aud), 2), round(corr(avg_median_income_aud, claims_per_1000_residents), 2)
FROM s CROSS JOIN ext WHERE s.state_code <> ext.top_loss
UNION ALL
SELECT 3, concat('without ', first(top_income), ' (highest income)'), count(*),
       round(corr(avg_median_income_aud, loss_per_resident_aud), 2), round(corr(avg_median_income_aud, claims_per_1000_residents), 2)
FROM s CROSS JOIN ext WHERE s.state_code <> ext.top_income
ORDER BY ord
""".format(t=otname("bq5_income_vs_loss")))

**How to read r:** +1 means higher income goes with higher loss per resident, −1 the opposite, 0 no straight-line relationship. The underinsurance idea predicts a **positive** r (poorer states insure less, so less insured loss). If r is negative or swings a lot when one state is removed, the data does not support it.